In [ ]:
import tkinter as tk
import random
import time
from collections import deque
GOAL   = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE   = 90
GAP    = 5
ANIM   = 500

ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
ANAMES  = {(-1,0):"UP",(1,0):"DOWN",(0,-1):"LEFT",(0,1):"RIGHT"}

VARIABLES = list(range(9))
DOMAIN    = list(range(9))

def misplaced(b):
    return sum(1 for i,v in enumerate(b) if v!=0 and v!=GOAL[i])

def apply_action(b, a):
    idx=b.index(0); r,c=divmod(idx,3); dr,dc=a; nr,nc=r+dr,c+dc
    if 0<=nr<3 and 0<=nc<3:
        ni=nr*3+nc; t=list(b); t[idx],t[ni]=t[ni],t[idx]; return tuple(t)
    return None

def count_conflicts(var, value, assignment):
    c = 0
    for j in VARIABLES:
        if j != var and assignment[j] == value:
            c += 1
    if value != GOAL[var]:
        c += 1
    return c

def conflicted_variables(assignment):
    conflicted = []
    for i in VARIABLES:
        if count_conflicts(i, assignment[i], assignment) > 0:
            conflicted.append(i)
    return conflicted

def min_conflicts(max_steps=100_000):
    values = list(range(9))
    random.shuffle(values)
    assignment = {i: values[i] for i in VARIABLES}
    log = []  
    for step in range(1, max_steps + 1):
        conflicted = conflicted_variables(assignment)
        if not conflicted:
            return assignment, step, log   
        var = random.choice(conflicted)
        conflict_counts = [
            (count_conflicts(var, v, assignment), v)
            for v in DOMAIN
        ]
        min_c = min(c for c, _ in conflict_counts)
        best_values = [v for c, v in conflict_counts if c == min_c]
        chosen = random.choice(best_values)
        log.append((step, var, assignment[var], chosen, min_c))
        assignment[var] = chosen

    return None, max_steps, log 
def bfs_path(start):
    parent  = {start: None}
    act_map = {start: None}
    q = deque([start])
    while q:
        s = q.popleft()
        if s == GOAL:
            path, acts = [], []
            while act_map[s] is not None:
                acts.append(ANAMES[act_map[s]]); path.append(s); s = parent[s]
            path.append(start); path.reverse(); acts.reverse()
            return path, acts
        for a in ACTIONS:
            nb = apply_action(s, a)
            if nb and nb not in parent:
                parent[nb]=s; act_map[nb]=a; q.append(nb)
    return [start], []

def minconflicts_solve(board, max_steps=100_000):
    assignment, steps, log = min_conflicts(max_steps)

    if assignment is None:
        return [], steps, [], 0, log

    solved_board = tuple(assignment[i] for i in range(9))
    if solved_board != GOAL:
        return [], steps, [], 0, log

    path_states, path_acts = bfs_path(board)
    return path_states, steps, path_acts, len(path_states)-1, log
def is_solvable(b):
    arr=[x for x in b if x!=0]
    inv=sum(1 for i in range(len(arr)) for j in range(i+1,len(arr)) if arr[i]>arr[j])
    return inv%2==0

def random_board():
    b=list(GOAL)
    while True:
        random.shuffle(b); t=tuple(b)
        if is_solvable(t) and t!=GOAL: return t

class App:
    def __init__(self, root):
        self.root=root
        self.root.title("8 Puzzle — CSP: Min-Conflicts ")
        self.root.configure(bg="white")
        self.root.resizable(False,False)

        self.board  =random_board()
        self.path   =[]
        self.moves  =[]
        self.log    =[]
        self.step   =0
        self.playing=False

        main=tk.Frame(root,bg="white"); main.pack(padx=10,pady=10)
        left=tk.Frame(main,bg="white"); left.pack(side="left",padx=10)

        self.canvas=tk.Canvas(left,width=3*TILE+4*GAP,height=3*TILE+4*GAP,
                              bg="white",highlightthickness=0)
        self.canvas.pack()

        self.lbl_step=tk.Label(left,text="",font=("Arial",12),bg="white")
        self.lbl_step.pack(pady=4)

        self.lbl_h=tk.Label(left,text="h(n) = —",bg="white",
                            fg="#c05000",font=("Arial",10))
        self.lbl_h.pack()

        nf=tk.Frame(left,bg="white",bd=1,relief="groove")
        nf.pack(pady=4,fill="x",padx=4)
        tk.Label(nf,text="Phase:",bg="white",
                 font=("Arial",9,"bold")).pack(side="left",padx=4)
        self.lbl_ntype=tk.Label(nf,text="—",bg="white",
                                fg="#0055cc",font=("Consolas",10))
        self.lbl_ntype.pack(side="left",padx=4)

        r1=tk.Frame(left,bg="white"); r1.pack(pady=2)
        self.lbl_nodes=tk.Label(r1,text="Steps: —",bg="white")
        self.lbl_nodes.grid(row=0,column=0,padx=5)
        self.lbl_steps=tk.Label(r1,text="Moves: —",bg="white")
        self.lbl_steps.grid(row=0,column=1,padx=5)
        self.lbl_time =tk.Label(r1,text="Time: — ms",bg="white")
        self.lbl_time.grid(row=0,column=2,padx=5)

        r2=tk.Frame(left,bg="white"); r2.pack(pady=2)
        self.lbl_depth=tk.Label(r2,text="Depth: —",bg="white",
                                fg="#006400",font=("Arial",10))
        self.lbl_depth.grid(row=0,column=0,padx=5)
        self.lbl_path=tk.Label(r2,text="P=[S]",bg="white",
                               fg="#00008b",font=("Arial",10))
        self.lbl_path.grid(row=0,column=1,padx=5)

        btns=tk.Frame(left,bg="white"); btns.pack(pady=6)
        tk.Button(btns,text="Shuffle",width=10,command=self.shuffle).grid(row=0,column=0,padx=4)
        tk.Button(btns,text="Reset",  width=10,command=self.reset  ).grid(row=0,column=1,padx=4)
        tk.Button(btns,text="Solve",  width=10,command=self.solve  ).grid(row=0,column=2,padx=4)
        tk.Button(btns,text="Step ▶",width=34,command=self.next_step).grid(
            row=1,column=0,columnspan=3,pady=4)

        right=tk.Frame(main,bg="white"); right.pack(side="right",padx=10)
        tk.Label(right,text="Solution (Min-Conflicts)",
                 font=("Arial",13,"bold"),bg="white").pack()
        self.sol=tk.Text(right,width=32,height=25,font=("Consolas",9))
        self.sol.pack()
        self.draw()

    def cur_board(self):
        if self.path and self.step<len(self.path): return self.path[self.step]
        return self.board

    def draw(self):
        self.canvas.delete("all")
        b=self.cur_board()
        self.lbl_h.config(text=f"h(n) = {misplaced(b)}")
        for i,val in enumerate(b):
            r,c=divmod(i,3)
            x1=GAP+c*(TILE+GAP); y1=GAP+r*(TILE+GAP)
            x2=x1+TILE;          y2=y1+TILE
            if val==0:
                self.canvas.create_rectangle(x1,y1,x2,y2,fill="#e8f5e9",
                    outline="#aaa",width=2,dash=(4,3))
            else:
                wrong=val!=GOAL[i]
                self.canvas.create_rectangle(x1,y1,x2,y2,
                    fill="#ffe0e0" if wrong else "#f5f5f5",
                    outline="#cc0000" if wrong else "#999",width=2)
                self.canvas.create_text((x1+x2)//2,(y1+y2)//2,
                    text=str(val),font=("Arial",28),
                    fill="red" if wrong else "black")

    def _reset(self):
        self.path=[];self.moves=[];self.log=[]
        self.step=0;self.playing=False
        for lbl,txt in [(self.lbl_nodes,"Steps: —"),(self.lbl_steps,"Moves: —"),
                        (self.lbl_time,"Time: — ms"),(self.lbl_depth,"Depth: —"),
                        (self.lbl_path,"P=[S]"),(self.lbl_step,""),(self.lbl_h,"h(n) = —")]:
            lbl.config(text=txt)
        self.lbl_ntype.config(text="—",fg="#0055cc")
        self.sol.delete("1.0",tk.END)

    def shuffle(self): self.board=random_board();self._reset();self.draw()
    def reset(self):   self.board=GOAL;self._reset();self.draw()

    def _path_str(self):
        L="SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        n=min(self.step+1,len(L))
        return "P=["+ ",".join(L[i] for i in range(n))+"]"

    def solve(self):
        if self.playing: return
        self._reset(); self.draw()
        t0=time.perf_counter()
        path_states,csp_steps,path_acts,depth,log=minconflicts_solve(self.board)
        ms=(time.perf_counter()-t0)*1000

        self.path=path_states; self.moves=path_acts
        self.log=log; self.step=0

        solved=bool(path_states) and path_states[-1]==GOAL
        self.lbl_nodes.config(text=f"Steps: {csp_steps}")
        self.lbl_steps.config(text=f"Moves: {depth}")
        self.lbl_time .config(text=f"Time: {ms:.1f} ms")
        self.lbl_depth.config(text=f"Depth: {depth}")
        self.lbl_step .config(text="✓ SOLVED" if solved else "✗ Failed")
        self.sol.delete("1.0",tk.END)
        self.sol.insert(tk.END,"=== Min-Conflicts Iterations ===\n")
        self.sol.insert(tk.END,f"  Total CSP steps: {csp_steps}\n\n")
        self.sol.insert(tk.END,"  step  var  old→new  min_c\n")
        self.sol.insert(tk.END,"  " + "─"*28 + "\n")
        shown = log[:15]
        for (step,var,old,new,mc) in shown:
            self.sol.insert(tk.END,
                f"  {step:4d}  X{var}   {old}→{new}      c={mc}\n")
        if len(log)>15:
            self.sol.insert(tk.END,f"  ...({len(log)} total iterations)\n")
        self.sol.insert(tk.END,"\n=== Move Sequence (BFS) ===\n")

        if path_states:
            self.playing=True
            self.lbl_ntype.config(text="Min-Conflicts",fg="#006600")
            self.animate()

    def _update_sol(self):
        if not self.path or self.step==0: return
        content=self.sol.get("1.0",tk.END)
        marker="=== Move Sequence (BFS) ===\n"
        idx=content.find(marker)
        if idx>=0:
            lb=content[:idx+len(marker)].count("\n")
            self.sol.delete(f"{lb+1}.0",tk.END)
        L="SABCDEFGHIJKLMNOPQRSTUVWXYZ"
        for i in range(1,self.step+1):
            m=self.moves[i-1] if i-1<len(self.moves) else "?"
            h=misplaced(self.path[i])
            nm=L[i]   if i  <len(L) else f"N{i}"
            pv=L[i-1] if i-1<len(L) else f"N{i-1}"
            self.sol.insert(tk.END,f"{i:2}. {pv} →{m:<6} h={h}  [{nm}]\n")
        self.sol.tag_remove("hl","1.0",tk.END)
        total=int(self.sol.index("end-1c").split(".")[0])
        self.sol.tag_add("hl",f"{total}.0",f"{total}.end")
        self.sol.tag_config("hl",background="#cce5ff")
        self.sol.see(tk.END)
        self.lbl_path.config(text=self._path_str())

    def animate(self):
        if not self.playing: return
        self.draw()
        if self.step==0:
            self.lbl_step.config(text="START"); self.lbl_path.config(text="P=[S]")
        else:
            m=self.moves[self.step-1] if self.step-1<len(self.moves) else "?"
            self.lbl_step.config(text=f"Step {self.step}: {m}")
        self._update_sol()
        if self.step<len(self.path)-1:
            self.step+=1; self.root.after(ANIM,self.animate)
        else:
            self.playing=False
            self.lbl_step.config(text="✓ SOLVED!" if self.path[-1]==GOAL else "✗ Stuck")

    def next_step(self):
        self.playing=False
        if not self.path or self.step>=len(self.path)-1: return
        self.step+=1; self.draw()
        m=self.moves[self.step-1] if self.step-1<len(self.moves) else "?"
        self.lbl_step.config(text=f"Step {self.step}: {m}")
        self._update_sol()

if __name__=="__main__":
    root=tk.Tk(); App(root); root.mainloop()